# AI Agents Workshop — Day 2 Labs (Colab)

S4DS KJSIT. MCP half (before the break).

**Before anything:** 🔑 Secrets → `HF_TOKEN` → toggle Notebook access on.

Hosted Mindicator MCP (default):
`https://personal-mindicatormcp.qbegzg.easypanel.host/mcp`


## Setup — run this once


In [ ]:
!pip install -q "huggingface_hub>=0.30.0" "smolagents[mcp]>=1.14.0" "mcp>=1.0.0" "python-dotenv>=1.0.0"

import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
MODEL_ID = "Qwen/Qwen2.5-72B-Instruct"
MINDICATOR_MCP_URL = os.environ.get(
    "MINDICATOR_MCP_URL",
    "https://personal-mindicatormcp.qbegzg.easypanel.host/mcp",
)

from huggingface_hub import InferenceClient
client = InferenceClient(model=MODEL_ID, token=os.environ["HF_TOKEN"])
r = client.chat.completions.create(
    messages=[{"role": "user", "content": "Reply with exactly: pong"}],
    max_tokens=10,
)
print("model said:", r.choices[0].message.content)
print("MCP URL:", MINDICATOR_MCP_URL)
print("SETUP OK" if "pong" in r.choices[0].message.content.lower() else "SETUP FAILED")


---
## Lab 0 — Ping Mindicator MCP

Day 1 tools lived in your file. Day 2 tools live behind a URL.


In [ ]:
from smolagents import MCPClient

mcp_config = {"url": MINDICATOR_MCP_URL, "transport": "streamable-http"}

with MCPClient(mcp_config, structured_output=True) as tools:
    print(f"{len(tools)} tools:")
    by_name = {}
    for t in tools:
        by_name[t.name] = t
        desc = (t.description or "").strip().split("")[0]
        print(f"- {t.name}: {desc}")
    print("health_check →")
    print(by_name["health_check"]())


**Try it:** break the URL (add a typo). What error do you see?

---
## Lab 1 — What MCP is

Same idea as a Day 1 tool — description the model reads — but the function runs on a server.


In [ ]:
import inspect

def get_weather(city: str) -> str:
    """Get weather for an Indian city. Use for temperature / rain / humidity."""
    return {"pune": "27C, clear"}.get(city.lower(), "unknown")

print("IN-PROCESS (Day 1 style):")
print(f"- {get_weather.__name__}{inspect.signature(get_weather)}: {(get_weather.__doc__ or '').split(chr(10))[0]}")
print("runs in: this notebook process")
print()
print("MCP (Day 2 style): tools listed above — runs on Mindicator host")
print("Your agent still does Thought → Action → Observation. Only the plug changed.")


---
## Lab 2 — Mindicator agent (agentic SQL)

`execute_sql` is retrieval-as-a-tool: the model decides what to fetch.


In [ ]:
from smolagents import InferenceClientModel, MCPClient, ToolCallingAgent

model = InferenceClientModel(model_id=MODEL_ID, token=os.environ["HF_TOKEN"])
TASK = (
    "How do I get from Churchgate to Thane on the local train? "
    "Also give a rough ticket fare if available. A few short sentences."
)

with MCPClient(mcp_config, structured_output=True) as tools:
    agent = ToolCallingAgent(tools=tools, model=model, max_steps=8)
    print(agent.run(TASK))


**Try it:**
1. `"What is the auto rickshaw fare for about 5 km at night?"`
2. `"Is train 95338 running late right now?"`
3. Add `verbosity_level=2` and watch `get_schema` / `execute_sql`.

---
## After the break

Open `notebooks/day2_lora.ipynb` for the LoRA SFT lab.

Take-home: `projects/project-2.md` — compose **2+ MCP servers**.
